# 07b - Diagnostics (no GPU)

Reads `history.csv` only. Nothing is trained, nothing is downloaded, no repo
checkout is needed. Run this on a **CPU runtime** so it costs no compute units.

It answers three things:

1. **What exists** - which rungs and seeds finished, and whether `L5_full` has
   aggregate results yet.
2. **Did the weak-stream fix change anything** - old vs new `L3_weak` compared at
   the *same epoch number*, which separates the effect of the fix from the fact
   that the best epoch moved earlier.
3. **Was the selected checkpoint overfitted** - which epoch each selection rule
   would pick, and how far the selected epoch's validation loss is above the
   minimum that run reached.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================================
# 7b - Diagnostics. No GPU needed, reads history.csv only.
# ============================================================================
import os, pandas as pd

DRIVE_BASE = '/content/drive/MyDrive/hari/SymptoGraph-LLM'
V5     = f'{DRIVE_BASE}/proposed'          # notebook 06 run (L0-L2 valid)
STREAM = f'{DRIVE_BASE}/proposed_stream'   # notebook 07 run (L3-L5 new)
SEEDS  = (0, 1, 2)
LADDER = ['L0_flat', 'L1_bottleneck', 'L2_graph',
          'L3_weak', 'L4_contrastive', 'L5_full']
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', None)

def hist(root, rung, seed):
    p = f'{root}/{rung}/seed_{seed}/history.csv'
    return pd.read_csv(p) if os.path.isfile(p) else None

# ---------------------------------------------------------------- 0. inventory
print('=' * 78)
print('0. WHAT EXISTS  (did L5 finish?)')
print('=' * 78)
inv = []
for root, tag in ((V5, 'v5'), (STREAM, 'stream')):
    for rung in LADDER:
        agg = os.path.isfile(f'{root}/{rung}/aggregate/aggregate_metrics.json')
        eps = []
        for s in SEEDS:
            h = hist(root, rung, s)
            eps.append(0 if h is None else len(h))
        if any(eps) or agg:
            inv.append({'run': tag, 'rung': rung,
                        'epochs_seed0': eps[0], 'epochs_seed1': eps[1],
                        'epochs_seed2': eps[2], 'has_aggregate': agg})
inv = pd.DataFrame(inv)
display(inv.set_index(['run', 'rung']))
missing = inv[(inv.run == 'stream') & (~inv.has_aggregate)]
if len(missing):
    print('INCOMPLETE in proposed_stream:', list(missing.rung))
    print('Re-run section 6 to resume those; finished rungs are skipped.')
else:
    print('All stream rungs have aggregate results.')

# ------------------------------------------- 1. old vs new L3, matched epochs
print()
print('=' * 78)
print('1. DID THE FIX CHANGE ANYTHING?  old vs new L3_weak at the SAME epoch')
print('=' * 78)
METRICS = ['val_composite', 'val_severity_risk_qwk',
           'val_severity_depression_qwk', 'val_group_all_macro_auroc',
           'val_group_all_macro_f1', 'val_loss']
pairs = []
for s in SEEDS:
    o, n = hist(V5, 'L3_weak', s), hist(STREAM, 'L3_weak', s)
    if o is None or n is None:
        continue
    cols = ['epoch'] + [m for m in METRICS if m in o.columns and m in n.columns]
    pairs.append(o[cols].merge(n[cols], on='epoch', suffixes=('_old', '_new')))
if not pairs:
    print('Need L3_weak history in BOTH folders to compare.')
else:
    m = pd.concat(pairs).groupby('epoch').mean()
    for met in METRICS:
        a, b = f'{met}_old', f'{met}_new'
        if a not in m or b not in m:
            continue
        t = m[[a, b]].copy()
        t['delta'] = t[b] - t[a]
        print(f'\n{met}   (mean over seeds)')
        print(t.round(4).to_string())

# ------------------------------------ 2. is the selected checkpoint overfitted?
print()
print('=' * 78)
print('2. WHICH EPOCH DID SELECTION PICK, AND WAS IT OVERFITTED?')
print('=' * 78)
rows = []
for root, tag in ((V5, 'v5'), (STREAM, 'stream')):
    for rung in LADDER:
        for s in SEEDS:
            h = hist(root, rung, s)
            if h is None or 'val_selection_score' not in h.columns:
                continue
            sel = h.loc[h.val_selection_score.idxmax()]
            lo  = h.loc[h.val_loss.idxmin()]
            rec = {'run': tag, 'rung': rung, 'seed': s, 'epochs': len(h),
                   'ep_selected': int(sel.epoch),
                   'ep_min_val_loss': int(lo.epoch),
                   'val_loss_selected': round(sel.val_loss, 3),
                   'val_loss_min': round(lo.val_loss, 3),
                   'inflation': round(sel.val_loss / lo.val_loss, 2)}
            if 'val_composite' in h.columns:
                rec['ep_max_composite'] = int(h.loc[h.val_composite.idxmax()].epoch)
            if 'val_severity_risk_qwk' in h.columns:
                rec['riskQWK_selected'] = round(sel.val_severity_risk_qwk, 3)
                rec['riskQWK_best'] = round(h.val_severity_risk_qwk.max(), 3)
            if 'train_severity_risk_qwk' in h.columns:
                rec['train_riskQWK_last'] = round(h.train_severity_risk_qwk.iloc[-1], 3)
            rows.append(rec)
diag = pd.DataFrame(rows)
display(diag.set_index(['run', 'rung', 'seed']))

bad = diag[diag.inflation > 1.3]
print(f'\n{len(bad)} of {len(diag)} runs selected a checkpoint whose validation '
      f'loss is >1.3x the minimum it reached.')
if len(bad):
    print(bad[['run', 'rung', 'seed', 'ep_selected', 'inflation']].to_string(index=False))
    print('\nselection_score = mean(symptom macro F1, symptom macro AUROC, '
          'severity QWK).')
    print('Two of three terms are symptom metrics, which keep climbing while the')
    print('ordinal heads memorise their small training sets - so the score rises')
    print('on epochs you would not ship. train_riskQWK_last near 1.000 is that')
    print('memorisation: 340 training rows, fitted perfectly.')
out = f'{STREAM}/selection_diagnostics.csv'
diag.to_csv(out, index=False)
print('\nsaved ->', out)
